In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [2]:
df = pd.read_csv(
    '../dados/dados_guilherme.csv',
    parse_dates=['data_solicitacao', 'datahora_consulta']
)

In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1649489 entries, 0 to 1649488
Data columns (total 17 columns):
 #   Column                   Non-Null Count    Dtype         
---  ------                   --------------    -----         
 0   idsolicitacao            1649489 non-null  int64         
 1   idpaciente               1649489 non-null  int64         
 2   renda_media_winsor       1379710 non-null  float64       
 3   situacao_solicitacao     1649489 non-null  str           
 4   corregulador             1649362 non-null  str           
 5   especialidade_mae        1649489 non-null  str           
 6   cid                      1649489 non-null  str           
 7   cnes_solicitante         1649489 non-null  int64         
 8   cnesexecutante           1649489 non-null  int64         
 9   situacao_agenda          1649489 non-null  str           
 10  possui_unidade_indicada  1649489 non-null  str           
 11  cid_ajustada             1649489 non-null  str           
 12  data_solici

In [4]:
date_cols = ['data_solicitacao', 'datahora_consulta']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

        df[col + '_year'] = df[col].dt.year
        df[col + '_month'] = df[col].dt.month
        df[col + '_dayofweek'] = df[col].dt.dayofweek

df = df.drop(
    columns=[c for c in date_cols if c in df.columns]
)

In [11]:
cat_cols = X.select_dtypes(
    include=['object', 'str', 'category', 'string', 'bool']
).columns.tolist()

num_cols = X.select_dtypes(include=['number']).columns.tolist()

In [5]:
X = df.drop(columns=[
    'target',
    'idsolicitacao',
    'idpaciente',
    'situacao_agenda',
    'situacao_solicitacao'
], errors='ignore')

In [6]:
y = df['target'].astype(int)

In [7]:
'''
cat_cols = X.select_dtypes(
    include=['object', 'str', 'category', 'string', 'bool']
).columns.tolist()

num_cols = X.select_dtypes(
    include=['number']
).columns.tolist()

# Preprocess categorical features
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

# Preprocess numerical features
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('cat', cat_pipeline, cat_cols),
    ('num', num_pipeline, num_cols),
])
'''

"\ncat_cols = X.select_dtypes(\n    include=['object', 'str', 'category', 'string', 'bool']\n).columns.tolist()\n\nnum_cols = X.select_dtypes(\n    include=['number']\n).columns.tolist()\n\n# Preprocess categorical features\ncat_pipeline = Pipeline([\n    ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),\n    ('encoder', OneHotEncoder(handle_unknown='ignore')),\n])\n\n# Preprocess numerical features\nnum_pipeline = Pipeline([\n    ('imputer', SimpleImputer(strategy='median')),\n    ('scaler', StandardScaler()),\n])\n\npreprocessor = ColumnTransformer([\n    ('cat', cat_pipeline, cat_cols),\n    ('num', num_pipeline, num_cols),\n])\n"

In [12]:
preprocessor = ColumnTransformer(
    [
        ('cat', Pipeline([
            ('imputer', SimpleImputer(
                strategy='constant',
                fill_value='MISSING'
            )),
            ('encoder', OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )),
        ]), cat_cols),

        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), num_cols),
    ],
    sparse_threshold=0,  # Force dense output
)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [16]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [17]:
gnb = GaussianNB()
gnb.fit(X_train_encoded, y_train)
y_pred = gnb.predict(X_test_encoded)

In [18]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9976568515116794
